<a href="https://colab.research.google.com/github/raulmckay-blip/full-rag-ui./blob/main/Full_RAG_UI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Cell 0: System Setup & Engine Start


This cell builds the foundation. We install the "eyes" (OCR tools like Tesseract) and the "brain" (LlamaIndex & Gemma 2). Crucially, we also launch a private local server in the background and "warm it up" with a dummy request. This ensures that the model is loaded into GPU memory immediately, so your first real question feels instant.

In [ ]:
import subprocess
import time
import os

start_time = time.time()

print("Installing system tools...")
print("=" * 60)

!apt-get update -qq && apt-get install -y -qq tesseract-ocr poppler-utils

print("Installing Python dependencies...")
!pip install -q \
    llama-index-core \
    llama-index-llms-ollama \
    llama-index-embeddings-huggingface \
    llama-index-readers-file \
    python-dotenv \
    pymupdf \
    pytesseract \
    pdf2image \
    easyocr \
    opencv-python-headless \
    "gradio>=5.0.0,<6.0.0"

print("=" * 60)
print(" Configuring Ollama server...")
os.environ["OLLAMA_KEEP_ALIVE"] = "-1"
os.environ["OLLAMA_HOST"] = "0.0.0.0"

!curl -fsSL https://ollama.com/install.sh | sh
process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    env=os.environ.copy()
)

time.sleep(5)
print("=" * 60)
print(" Downloading and warming up Gemma 2 (2B)...")
!ollama pull gemma2:2b

!curl http://localhost:11434/api/generate -d '{"model": "gemma2:2b", "prompt": "Init", "stream": false}' > /dev/null 2>&1

elapsed = int(time.time() - start_time)
print("=" * 60)
print(f"Setup complete in {elapsed} seconds. Environment ready.")

#Cell 0.5: Intelligence Configuration

Here, we define the "brain" of the operation. We configure Gemma 2 to act as a strict analyst by setting the temperature to 0.1, which minimizes creativity and prevents hallucinations. We also load the Embedding Model (BGE-Small), which translates human text into mathematical vectors. This specific model was chosen because it is highly efficient on local hardware while maintaining high accuracy for English retrieval.

In [ ]:
import os
import torch
from llama_index.core import Settings
from llama_index.llms.ollama import Ollama
from llama_index.embeddings.huggingface import HuggingFaceEmbedding


os.environ["OLLAMA_KEEP_ALIVE"] = "-1"

LLM_MODEL_NAME = "gemma2:2b"
EMBED_MODEL_NAME = "BAAI/bge-small-en-v1.5"
OLLAMA_URL = "http://localhost:11434"
PERSIST_DIR = "./storage_mini"
CHUNK_SIZE = 512
CHUNK_OVERLAP = 50

print(f"Loading Intelligence Model: {LLM_MODEL_NAME}...")

Settings.llm = Ollama(
    model=LLM_MODEL_NAME,
    base_url=OLLAMA_URL,
    request_timeout=300.0,
    temperature=0.1,
    additional_kwargs={"num_ctx": 4096}
)

Settings.embed_model = HuggingFaceEmbedding(
    model_name=EMBED_MODEL_NAME,
    device="cuda" if torch.cuda.is_available() else "cpu"
)
print("=" * 60)
print("Model loaded and configuration active.")

# Cell 1: Computer Vision & OCR Layer
This is the "vision" component of the pipeline. Instead of relying on standard text extraction, we employ a "Force-OCR" strategy to handle scanned or handwritten documents. We use OpenCV to apply CLAHE (adaptive contrast enhancement), which separates faint ink from the paper; crucial for distinguishing handwritten dates (like "June 20" vs "June 28"). We then feed these enhanced images into EasyOCR, running on the GPU for maximum accuracy.

In [ ]:
import fitz
from PIL import Image
import io
import easyocr
import numpy as np
import cv2

print(" Initializing OCR Engine...")
reader = easyocr.Reader(['en'], gpu=True)

def preprocess_image_clahe(pil_image):
    """
    Applies CLAHE (Contrast Limited Adaptive Histogram Equalization) to
    enhance ink contrast and separate dense handwriting text (e.g., '0' vs ',').
    """
    img_np = np.array(pil_image)

    gray = cv2.cvtColor(img_np, cv2.COLOR_RGB2GRAY)

    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    enhanced = clahe.apply(gray)

    _, binary = cv2.threshold(enhanced, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    return binary

def run_ocr_on_page(page, zoom_level=3.0):
    """
    Renders a PDF page to image at specified zoom and runs OCR.
    Zoom level 3.0 is optimized for handwritten text clarity.
    """
    pix = page.get_pixmap(matrix=fitz.Matrix(zoom_level, zoom_level))
    img_data = pix.tobytes("png")
    pil_image = Image.open(io.BytesIO(img_data))

    processed_img_np = preprocess_image_clahe(pil_image)

    results = reader.readtext(processed_img_np, detail=0, paragraph=True)
    return " ".join(results)

def extract_text_with_hybrid_ocr(file_input):
    """
    Main extraction function. Handles file inputs and iterates through pages,
    applying the CLAHE preprocessing pipeline.
    """
    try:
        if isinstance(file_input, list):
            file_path = file_input[0].name
        elif hasattr(file_input, 'name'):
            file_path = file_input.name
        else:
            file_path = file_input

        doc = fitz.open(file_path)
        pages_data = []
        file_name = file_path.split('/')[-1]

        print(f"Processing file: {file_name}")

        for page_index, page in enumerate(doc):
            page_num = page_index + 1
            text = ""

            try:
                print(f"    - Page {page_num}: Scanning...")
                text = run_ocr_on_page(page, zoom_level=3.0)
            except Exception as e:
                print(f"    ! Page {page_num} Error: {e}")

            snippet = text.replace('\n', ' ')[:50]
            if len(snippet) > 0:
                print(f"      Extracted: \"{snippet}...\"")

            pages_data.append({"text": text, "page_num": page_num})

        doc.close()
        return pages_data

    except Exception as e:
        print(f"Critical Error: {e}")
        return []
print("=" * 60)
print(" EasyOCR Engine ready!")



# Cell 2: Smart Chunking & Metadata Enrichment
Raw text is difficult for an AI to navigate, so we need to organize it. In this step, we classify the document (e.g., "Invoice" vs. "Contract") and break it down into smaller, manageable "chunks" of text. Crucially, we stamp every single chunk with specific metadata; including its origin filename and page number. This "digital footprint" is what allows the final chatbot to provide precise citations like "Found on Page 3" rather than just giving a generic answer.

In [ ]:
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core.schema import TextNode

if 'CHUNK_SIZE' not in globals():
    print("[Config] CHUNK_SIZE not found. Using default: 512")
    CHUNK_SIZE = 512
    CHUNK_OVERLAP = 50

def determine_doc_type(text_sample):
    """
    Classifies the document based on keywords in the initial text.
    Used to provide context to the LLM (e.g., handling Invoices vs Contracts).
    """
    text_lower = text_sample.lower()[:2000]

    if "invoice" in text_lower or "bill to" in text_lower or "total amount" in text_lower:
        return "Invoice"
    elif "contract" in text_lower or "agreement" in text_lower or "terms" in text_lower:
        return "Legal Contract"
    elif "resume" in text_lower or "experience" in text_lower:
        return "Resume/CV"
    elif "abstract" in text_lower and "introduction" in text_lower:
        return "Research Paper"
    return "General Document"

def create_nodes_with_metadata(pages_data, file_name="Uploaded Doc"):
    """
    Converts raw page text into structured LlamaIndex Nodes with metadata
    (Page Number, Document Type, Filename) for accurate citations.
    """
    if not pages_data:
        return []

    intro_text = " ".join([p['text'] for p in pages_data[:2]])
    doc_type = determine_doc_type(intro_text)

    parser = SentenceSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)
    nodes = []

    for page in pages_data:
        text = page['text']
        page_num = page['page_num']

        if not text.strip():
            continue

        chunks = parser.split_text(text)

        for chunk in chunks:
            node = TextNode(text=chunk)

            node.metadata = {
                "source_identifier": file_name,
                "page_label": str(page_num),
                "document_type": doc_type,
            }

            node.excluded_llm_metadata_keys = []

            nodes.append(node)

    print(f"[Chunking] Generated {len(nodes)} segments. Type: {doc_type} | File: {file_name}")
    return nodes

# Cell 3: Vector Indexing & Storage Management
Here, we transform our text chunks into a searchable mathematical format called "Vector Embeddings." This effectively creates a high-speed map of the document's content. To optimize performance, we implement a persistence layer: once the index is built, we save it to the local disk. This allows us to reload the "memory" instantly in future sessions without re-running the computationally expensive embedding process.

In [ ]:
import os
from llama_index.core import VectorStoreIndex, StorageContext, load_index_from_storage

def build_vector_index(nodes, force_rebuild=False):
    """
    Manages the VectorStoreIndex lifecycle.
    Attempts to load a pre-built index from disk to save time.
    If 'force_rebuild' is True or no index exists, it generates a new one using GPU embeddings.
    """
    if not force_rebuild and os.path.exists(PERSIST_DIR):
        try:
            print(f"Found existing storage at {PERSIST_DIR}. Attempting load...")
            storage_context = StorageContext.from_defaults(persist_dir=PERSIST_DIR)

            index = load_index_from_storage(storage_context)
            print("Successfully loaded index from disk.")
            return index
        except Exception as e:
            print(f"Cache load failed ({e}). Proceeding to rebuild...")

    if not nodes:
        print("No text nodes provided for indexing.")
        return None

    print(f"Building new vector index from {len(nodes)} segments...")
    print("[Processing embeddings...")

    index = VectorStoreIndex(nodes)

    if not os.path.exists(PERSIST_DIR):
        os.makedirs(PERSIST_DIR)

    index.storage_context.persist(persist_dir=PERSIST_DIR)
    print(f"Index built and saved to {PERSIST_DIR} for future use.")

    return index

# Cell 4: The Intelligence Engine
This cell is the "Central Command" of our application. We define three critical functions here:

* process_pdfs: The conductor that triggers the OCR, builds the index, and initializes the Chat Engine. It includes a specific "System Prompt" that forces the AI to ignore distracting form codes (like "03/05") and focus only on real dates.

* chat_response: The logic that sends your question to the AI, retrieves the top 5 relevant document chunks, and generates an answer.

* format_intelligence_report: A formatter that takes the raw answer and appends the "Evidence Board"—listing exactly which page and paragraph the AI used to verify its claims.


In [ ]:
import gradio as gr

chat_engine = None
last_processed_files = []

def format_intelligence_report(query, response_object):
    """
    Formats the raw LLM response into a structured markdown report
    with citation support and source ranking.
    """
    ai_text = str(response_object)
    source_nodes = response_object.source_nodes

    formatted_output = (
        f"**1. Query:** {query}\n"
        f"**Response:**\n"
        f"{ai_text}\n\n"
        f"**Top Sources:**\n"
    )

    for i, node in enumerate(source_nodes[:2]):
        page_num = node.node.metadata.get('page_label', 'N/A')
        file_name = node.node.metadata.get('source_identifier', 'Unknown')
        score = round(node.score, 4) if node.score else 0.0000

        raw_text = node.node.get_content().replace('\n', ' ').strip()
        snippet = raw_text[:250] + "..." if len(raw_text) > 250 else raw_text

        formatted_output += (
            f"   • **Rank {i+1}** | Page {page_num} | Score {score}\n"
            f"     > \"{snippet}\"\n\n"
        )

    return formatted_output

def process_pdfs(pdf_files):
    """
    Orchestrates the document processing pipeline.
    Checks cache -> Runs OCR -> Chunks Text -> Builds Index.
    """
    global chat_engine, last_processed_files

    if not pdf_files:
        return [{"role": "assistant", "content": "Please upload a PDF file first."}]

    try:
        current_files = pdf_files if isinstance(pdf_files, list) else [pdf_files]
        current_filenames = sorted([f.name.split('/')[-1] for f in current_files])

        if chat_engine is not None and last_processed_files == current_filenames:
            print("[System] Cache hit. Skipping reprocessing.")
            return [{"role": "assistant", "content": "⚡ Ready! (Using cached index)"}]

        all_nodes = []
        print(f"[System] Processing {len(current_files)} new documents...")

        for file_obj in current_files:
            pages_data = extract_text_with_hybrid_ocr(file_obj)

            if pages_data:
                file_name = getattr(file_obj, 'name', 'doc').split('/')[-1]
                nodes = create_nodes_with_metadata(pages_data, file_name=file_name)
                all_nodes.extend(nodes)

        if not all_nodes:
            return [{"role": "assistant", "content": "Error: No text extracted. Please check the document quality."}]

        index = build_vector_index(all_nodes, force_rebuild=True)

        if index is None:
            return [{"role": "assistant", "content": "Critical Error: Index creation failed."}]

        system_prompt = (
            "You are a precision analyst. Answer the user's question using ONLY the provided context. "
            "If the document mentions a form number like '(0305)02', ignore it—it is NOT a date. "
            "Look for explicit dates written in text."
        )

        chat_engine = index.as_chat_engine(
            chat_mode="context",
            system_prompt=system_prompt,
            similarity_top_k=5
        )

        last_processed_files = current_filenames
        print("[System] Pipeline complete. Chat engine ready.")
        return [{"role": "assistant", "content": f"Insights unlocked. What would you like to know?"}]

    except Exception as e:
        last_processed_files = []
        print(f"[System] Critical Error: {e}")
        return [{"role": "assistant", "content": f"System Error: {str(e)}"}]


def add_user_message(history, message):
    """
    Updates the UI with the user's message immediately.
    """
    if message.strip() == "":
        return history, message
    history.append({"role": "user", "content": message})
    return history, ""

def chat_response(history):
    """
    Generates the AI response and applies formatting.
    """
    if not history:
        return history

    last_user_msg = history[-1]['content']

    if chat_engine is None:
        history.append({"role": "assistant", "content": "Please upload a PDF and click 'Analyze' first."})
        return history

    try:
        response = chat_engine.chat(last_user_msg)

        final_report = format_intelligence_report(last_user_msg, response)
        history.append({"role": "assistant", "content": final_report})

    except Exception as e:
        history.append({"role": "assistant", "content": f"Runtime Error: {str(e)}"})

    return history

# Cell 4.5: System Logic Test (Debug Mode)

Before launching the full application, we run a manual test to verify that the core pipeline is functioning correctly. This script forces a specific file through every stage - OCR, Indexing, and Retrieval — without the complexity of the graphical interface. It allows us to confirm that the AI can actually read the handwriting and find the answer.

Note: Ensure you have a file uploaded to the path defined in TEST_FILE before running this.

In [ ]:
import os
import sys
from IPython.display import Markdown, display

def run_debug_test(file_path, question):
    """
    Manually executes the full document pipeline on a specific file.
    Useful for validating OCR quality and Indexing logic without the UI.
    """
    if 'extract_text_with_hybrid_ocr' not in globals():
        print(" Critical Error: Functions from Cell 1/2/3 are missing.")
        print("         Please run previous cells first.")
        return

    print(f" Initiating validation run on: {file_path}")

    print(" Running OCR...")
    try:
        pages_data = extract_text_with_hybrid_ocr(file_path)
    except Exception as e:
        print(f" OCR execution failed: {e}")
        return

    if not pages_data:
        print(" OCR returned no data. Check file path/integrity.")
        return

    print(" Generatng metadata and text nodes...")
    clean_name = file_path.split("/")[-1]
    nodes = create_nodes_with_metadata(pages_data, file_name=clean_name)

    print(" Building Vector Index...")
    index = build_vector_index(nodes, force_rebuild=True)

    if not index:
        print(" Index creation failed.")
        return

    print(" Initializing Reasoning Engine...")

    system_prompt = (
        "You are a precision analyst. Answer the user's question using ONLY the provided context. "
        "If the document mentions a form number like '(0305)02', ignore it—it is NOT a date. "
        "Look for explicit dates written in text."
    )

    chat_engine = index.as_chat_engine(
        chat_mode="context",
        system_prompt=system_prompt,
        similarity_top_k=5
    )

    print(f"\nQuery: {question}\n")
    response = chat_engine.chat(question)

    print("="*60)
    print("AI RESPONSE:")
    print(response)
    print("\n" + "="*60)
    print("SOURCE EVIDENCE:")

    for i, node in enumerate(response.source_nodes):
        page = node.node.metadata.get('page_label', 'N/A')
        text_preview = node.node.get_content().replace('\n', ' ')[:200]
        score = round(node.score, 4) if node.score else 0.0000

        print(f"[{i+1}] Page {page} | Score: {score}")
        print(f"    > \"{text_preview}...\"\n")

TEST_FILE = "/content/MTG_10009588.pdf"

TEST_QUESTION = "Who is the borrower and exactly what date was the instrument acknowledged?"

if os.path.exists(TEST_FILE):
    run_debug_test(TEST_FILE, TEST_QUESTION)
else:
    print(f" File not found: {TEST_FILE}. Please upload it to the Files sidebar first.")

# Cell 5: The User Interface

Finally, we build the visual dashboard that brings the project to life. We use Gradio to create a clean, split-screen layout: the left sidebar handles document ingestion, while the main panel displays the chat interface. We inject custom CSS to enforce a professional color scheme and wire up the buttons to trigger the backend logic we defined in the previous cells.

In [ ]:
import gradio as gr

EXPENSIVE_WHITE, PRIMARY_COLOR, ACCENT_COLOR = "#F3F4F6", "#2D3436", "#6C5CE7"
SLATE_BUBBLE = "#3F4547"
my_theme = gr.themes.Soft(primary_hue="slate", neutral_hue="slate", spacing_size="sm", radius_size="lg")

custom_css = f".gradio-container, .dark, body {{ background-color: white !important; color: {PRIMARY_COLOR} !important; --neutral-900: #ffffff; }} #header h1 {{ color: {PRIMARY_COLOR}; font-family: 'Helvetica Neue', sans-serif; font-weight: 700; }} #sidebar, textarea, .gr-input {{ background-color: #f9fafb !important; border-color: #e5e7eb !important; color: {PRIMARY_COLOR} !important; }} #sidebar {{ border-right: 1px solid #e5e7eb; padding: 20px; }} #instructions, #instructions * {{ color: {PRIMARY_COLOR} !important; }} #chatbot {{ background-color: {PRIMARY_COLOR} !important; border: 1px solid #3c4043; }} #chatbot .message-bubble {{ color: {EXPENSIVE_WHITE} !important; border: none !important; }} #chatbot .message-row.bot-row .message-bubble {{ background-color: {SLATE_BUBBLE} !important; }} #chatbot .message-row.user-row .message-bubble {{ background-color: {ACCENT_COLOR} !important; }} #status-bar {{ padding: 8px 15px; border-radius: 6px; background-color: #e0f2fe !important; border-left: 5px solid {ACCENT_COLOR}; margin-bottom: 10px; }} #status-bar * {{ color: #0c4a6e !important; font-family: monospace; }} .gr-button-primary {{ background-color: {PRIMARY_COLOR} !important; color: {EXPENSIVE_WHITE} !important; box-shadow: 0 2px 4px rgba(0,0,0,0.1); }}"

with gr.Blocks(theme=my_theme, css=custom_css, title="PDF Assistant") as demo:

    with gr.Row():
        gr.Markdown("# 📄 PDF Assistant", elem_id="header")

    with gr.Row(equal_height=True):
        with gr.Column(scale=1, elem_id="sidebar"):
            pdf_input = gr.File(label="Select Documents", file_count="multiple", file_types=[".pdf"], height=300)
            process_btn = gr.Button("🔍 Analyze Documents", variant="primary", size="lg")

            gr.Markdown(
                "---\n### Instructions\n* **Upload** PDFs.\n* Click **Analyze**.\n* **Chat** to extract data.",
                elem_id="instructions"
            )

        with gr.Column(scale=2):
            status_bar = gr.Markdown("💤 System Idle - Waiting for documents...", elem_id="status-bar")
            chatbot = gr.Chatbot(elem_id="chatbot", label="AI Output", type="messages", height=600, show_copy_button=True, bubble_full_width=False)

            with gr.Row():
                query_input = gr.Textbox(placeholder="Ask a question...", scale=5, container=False, autofocus=True)
                send_btn = gr.Button("➤ Send", variant="primary", scale=1)

    process_btn.click(fn=lambda: "⏳ Analyzing...", outputs=[status_bar]).then(
        fn=process_pdfs, inputs=[pdf_input], outputs=[chatbot]
    ).then(fn=lambda: "✅ Ready! Ask a question below.", outputs=[status_bar])

    for trigger in [query_input.submit, send_btn.click]:
        trigger(
            fn=add_user_message, inputs=[chatbot, query_input], outputs=[chatbot, query_input]
        ).then(
            fn=chat_response, inputs=[chatbot], outputs=[chatbot]
        )
demo.queue().launch(share=True, debug=True)

# Cell 6: Pipeline Performance Benchmarking

In this final diagnostic step, we measure the "heartbeat" of the system. This script runs a controlled test to time each specific component—OCR (Vision), Indexing (Memory), and LLM (Reasoning)—independently. It then generates a professional bar chart and data table, providing the concrete performance metrics we need to justify our architecture choices in the final presentation.

In [ ]:
import time
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

def run_performance_benchmark(file_path, test_query):
    """
    Runs the pipeline components individually to measure latency and confidence.
    Does NOT affect the main chat interface.
    """
    print(f"📊 STARTING BENCHMARK ON: {file_path.split('/')[-1]}...")
    metrics = {}

    # OCR Speed
    start = time.time()
    try:
        pages_data = extract_text_with_hybrid_ocr(file_path)
        ocr_time = time.time() - start
        metrics['OCR (Vision)'] = round(ocr_time, 2)
        print(f"   ⏱️ OCR Time: {metrics['OCR (Vision)']}s")
    except Exception as e:
        print(f"❌ Benchmark Failed at OCR: {e}")
        return

    # Indexing Speed
    start = time.time()
    clean_name = file_path.split("/")[-1]
    nodes = create_nodes_with_metadata(pages_data, file_name=clean_name)
    index = build_vector_index(nodes, force_rebuild=True)
    indexing_time = time.time() - start
    metrics['Indexing (Storage)'] = round(indexing_time, 2)
    print(f"   ⏱️ Indexing Time: {metrics['Indexing (Storage)']}s")

    # Generation Speed & Quality
    chat_engine = index.as_chat_engine(
        chat_mode="context",
        system_prompt="Answer strictly based on context.",
        similarity_top_k=5
    )

    start = time.time()
    response = chat_engine.chat(test_query)
    gen_time = time.time() - start
    metrics['Thinking (LLM)'] = round(gen_time, 2)
    print(f"   ⏱️ LLM Time: {metrics['Thinking (LLM)']}s")

    top_score = response.source_nodes[0].score if response.source_nodes else 0
    quality_score = round(top_score * 100, 2)

    # Visualization
    print("\n" + "="*40)
    print("="*40)

    # Data Frame
    df = pd.DataFrame([
        {"Component": "OCR / Vision", "Time (sec)": metrics['OCR (Vision)'], "Role": "Extract Text"},
        {"Component": "Indexing", "Time (sec)": metrics['Indexing (Storage)'], "Role": "Embed & Store"},
        {"Component": "LLM / Thinking", "Time (sec)": metrics['Thinking (LLM)'], "Role": "Generate Answer"},
        {"Component": "Total Pipeline", "Time (sec)": sum(metrics.values()), "Role": "End-to-End"}
    ])
    display(df)
    print(f"\n🌟 Model Confidence Score: {quality_score}% (Relevance of retrieved text)")

    # Chart
    components = list(metrics.keys())
    times = list(metrics.values())

    plt.figure(figsize=(10, 6))
    bars = plt.bar(components, times, color=['#6C5CE7', '#00cec9', '#fdcb6e'])

    # Labels
    for bar in bars:
        yval = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2, yval + 0.1, f"{yval}s", ha='center', va='bottom', fontweight='bold')

    plt.title(f"Pipeline Latency Analysis\n", fontsize=14, fontweight='bold')
    plt.ylabel("Processing Time (Seconds)")
    plt.grid(axis='y', linestyle='--', alpha=0.7)

    plt.savefig("pipeline_metrics.png", dpi=300)
    plt.show()
    print("✅ Graph saved as 'pipeline_metrics.png' (Check Files tab)")


TEST_FILE_PATH = "/content/MTG_10009588.pdf"

TEST_QUERY_METRIC = "Who is the borrower?"

if os.path.exists(TEST_FILE_PATH):
    run_performance_benchmark(TEST_FILE_PATH, TEST_QUERY_METRIC)
else:
    print("⚠️ Please upload a PDF and set TEST_FILE_PATH first.")